# 7장. 작은 GPT 조립하기

임베딩, 블록 여러 층, 출력층을 합쳐 GPT를 완성해요. 이번에는 **책의 실제 크기**(약 600만 개 파라미터)로 만듭니다. 이 장은 **GPU가 필요 없어요.** (런타임 유형: CPU)

이 장에서 **손실(loss)**을 처음 계산해 봅니다.


### 준비
3장의 `Embeddings`, 6장의 `Block`을 패키지에서 불러오고, 이야기 200편을 번호로 바꿔 둡니다.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import math
import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab, GPTConfig, Embeddings, Block
from modu_llm import load_stories, encode_texts, get_batch

torch.manual_seed(42)
vocab = Vocab.build()
stories = load_stories(200)
data = encode_texts(stories, vocab)
print("글자 수:", len(data))


## 7.1 전체 구조 한눈에 보기

```
글자 번호 (B, T)
  │  Embeddings: 글자 임베딩 + 위치 임베딩        (3장)
  ▼
벡터 (B, T, 256)
  │  Block × 6: 어텐션 + MLP, LayerNorm, 잔차 연결   (4~6장)
  ▼
벡터 (B, T, 256)
  │  LayerNorm → Linear(출력층)
  ▼
다음 글자 점수 (B, T, 2453)   ← 사전의 모든 글자마다 점수 하나
```
B는 배치 크기, T는 글자 수예요. 마지막 출력층은 각 위치에서 **사전의 모든 글자에 점수**를 매깁니다.


## 7.2 GPT 클래스 완성하기

설정값을 `GPTConfig` 하나에 모았어요. 기본값이 책의 모델 크기입니다.

**핵심 코드**: `self.head = nn.Linear(cfg.n_embd, cfg.vocab_size)` — 글자마다 점수


In [ ]:
cfg = GPTConfig()
print(cfg)
print("사전 크기와 맞나요?", cfg.vocab_size == len(vocab))


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = Embeddings(cfg)
        self.blocks = nn.ModuleList()
        for i in range(cfg.n_layer):
            self.blocks.append(Block(cfg))
        self.ln = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size)

    def forward(self, idx, targets=None):
        x = self.emb(idx)
        for block in self.blocks:
            x = block(x)
        x = self.ln(x)
        logits = self.head(x)

        if targets is None:
            return logits, None
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))
        return logits, loss

model = TinyGPT(cfg)


## 7.3 다음 글자 점수와 손실

모델은 위치마다 2,453개 글자에 점수(logits)를 냅니다. 이 점수를 확률로 바꿨을 때 **정답 글자의 확률이 낮을수록 손실이 커요**(cross entropy).
학습 전 모델은 모든 글자를 비슷하게 보므로, 손실은 ln(2453) ≈ 7.8 근처에서 시작합니다.

**핵심 코드**: `F.cross_entropy(점수, 정답)`


In [ ]:
x, y = get_batch(data, cfg.block_size, 4)
logits, loss = model(x, y)
print("x:", x.shape)
print("logits:", logits.shape)

start_loss = loss.item()
guess_loss = math.log(len(vocab))
print(f"학습 전 손실: {start_loss:.2f}")
print(f"ln(사전 크기): {guess_loss:.2f}")


손실이 어떻게 계산되는지, 글자가 3개뿐인 작은 예로 확인해 봅니다. 정답은 0번 글자예요.


In [ ]:
scores = torch.tensor([[2.0, 1.0, 0.0]])
answer = torch.tensor([0])

probs = F.softmax(scores, dim=-1)
print("확률:", probs)
print("-ln(정답 확률):", -torch.log(probs[0, 0]))
print("cross_entropy:", F.cross_entropy(scores, answer))


## 7.4 파라미터 수 세어 보기

파라미터는 학습하며 고쳐 나갈 숫자들이에요. 부분별로 세어 보면 어디에 가장 많은지 알 수 있습니다.


In [ ]:
def count(m):
    total = 0
    for p in m.parameters():
        total = total + p.numel()
    return total

n_emb = count(model.emb)
n_blocks = count(model.blocks)
n_out = count(model.ln) + count(model.head)
n_all = count(model)
print(f"임베딩: {n_emb:,}")
print(f"블록 6개: {n_blocks:,}")
print(f"출력층: {n_out:,}")
print(f"전체: {n_all:,}")


## 7.5 학습 전 GPT로 글 만들어 보기

생성 함수는 10장에서 자세히 다루고, 여기서는 패키지의 `generate`를 빌려 써요. 학습 전이라 사전의 글자가 무작위로 나옵니다.


In [ ]:
from modu_llm import generate

text = generate(model, vocab, "어느 날, ", max_new_tokens=80, stop_at_end=False)
print(text)


## 7.6 지금까지 만든 부품 한곳에 모으기

3~7장에서 만든 `Embeddings`, `Head`, `MultiHead`, `Block`, `TinyGPT`는 실습 패키지 `modu_llm`에 같은 내용으로 들어 있어요.
다음 장부터는 검증된 패키지 버전을 불러오므로, 직접 쓴 코드에 작은 실수가 있어도 막히지 않습니다. 직접 비교해 보세요.


In [ ]:
import inspect
import modu_llm

source = inspect.getsource(modu_llm.TinyGPT)
print(source)


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

층 수와 벡터 길이를 바꾸면 파라미터 수가 어떻게 달라지는지 보세요. 어느 쪽이 더 빨리 늘어나나요?


In [ ]:
for n_layer, n_embd in [(6, 256), (12, 256), (6, 512)]:
    small_cfg = GPTConfig(n_layer=n_layer, n_embd=n_embd)
    m = TinyGPT(small_cfg)
    n = count(m)
    print(f"층 {n_layer}, 벡터 {n_embd}: {n:,}")


## 실제 LLM에서는

- 이 책의 모델은 약 600만 개 파라미터예요. 실제 LLM은 수십억~수천억 개입니다.
- 실제 LLM은 출력층과 글자 임베딩이 같은 가중치를 함께 쓰게 해(weight tying) 파라미터를 아끼기도 해요.
